# Notebook 06b - Corpus RecipeML : modélisation SMT sur données externes

**Navigation** : [Index Z3](./README.md) | [Index SymbolicAI](../../README.md) | [Index général](../../../README.md) | [Notebook 05 (Meal Planner)](./05_Meal_Planner_Hierarchical.ipynb)

## Objectifs d'apprentissage

- Charger un corpus **structuré externe** au format **RecipeML** (standard XML culinaire) et le parser en classes de domaine C#
- Modéliser un **menu équilibré** sans allergène par sélection booléenne (variables Z3 une par recette) avecagrégation nutritionnelle conditionnelle via `MkITE`
- Étendre le modèle à un **plan multi-jours** hiérarchique en `int[][]` avec contrainte de non-répétition (`Z3Methods.Distinct`) sur les indices du corpus parsé
- Comparer l'**élégance déclarative** du DSL Z3.Linq face à l'énumération manuelle lorsque les données viennent d'une source externe

## Prérequis

- Notebook [05 (Meal Planner Hierarchical)](./05_Meal_Planner_Hierarchical.ipynb) : pattern `int[][]`, `MkITE`, `CollectionHandling.Array`
- Concepts : LINQ to XML (`XDocument`), classes/records C#, théorème hiérarchique Z3.Linq

**Durée estimée** : 35-45 minutes

In [1]:
#r "../Z3.Linq/.deploy/Microsoft.Z3.dll"
#r "../Z3.Linq/.deploy/ExpressionUtils.dll"
#r "../Z3.Linq/.deploy/Z3.Linq.dll"
using Z3.Linq;
using Microsoft.Z3;
using System;
using System.Collections.Generic;
using System.Linq;
using System.Xml.Linq;
Console.WriteLine("Imports OK : Z3.Linq (.deploy), Microsoft.Z3, System.Xml.Linq (RecipeML)");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Imports OK : Z3.Linq (.deploy), Microsoft.Z3, System.Xml.Linq (RecipeML)


---

## Introduction : pourquoi RecipeML ?

Jusqu'ici, les notebooks de la série Z3.Linq travaillaient sur des **données littérales définies dans le notebook lui-même** (une `List<FoodItem>` codée à la main dans la section 2 du notebook 05). En pratique, les données d'entrée d'un solveur de contraintes proviennent d'une **source externe** : base de données relationnelle, fichier CSV, API REST, ou — c'est le cas ici — un **document structuré au format standard**.

**RecipeML** (Recipe Markup Language) est un format XML ouvert conçu pour l'échange de recettes de cuisine. On le rencontre dans les catalogues de restauration, les applications de meal-planning et certaines bases nutritionnelles publiques. Sa structure hiérarchique (`<recipeml>` → `<recipe>` → `<head>`, `<nutrition>`, `<ingredients>`) se prête naturellement à un parsing `XDocument` puis à une projection vers des **classes de domaine** que le solveur Z3 va consommer.

### Ce que le solveur va démontrer

Le notebook montre deux capacités complémentaires du moteur SMT sur ce corpus externe :

1. **Sélection booléenne avec agrégation nutritionnelle conditionnelle** (section 3) — un `BoolExpr` par recette, somme pondérée via `MkITE`, contraintes de budget / protéines / exclusion d'allergène. C'est la même technique que le notebook 05 section 5, mais **alimentée par les valeurs issues du XML**.
2. **Plan hiérarchique multi-jours en `int[][]`** (section 4) — sélection d'indices de recettes dans le corpus parsé, avec **non-répétition** via `Z3Methods.Distinct` en mode `CollectionHandling.Array`. Le même mécanisme qui structure un Sudoku (notebook 04) ou un plan hebdomadaire (notebook 05 section 7) s'applique ici sans modification.

> **Note de lisibilité** : la section 4 regroupe dans une même cellule la déclaration des bornes, la construction du théorème `Where(...)` et l'appel `.Solve()`. C'est un choix de **présentation** (bornes et contraintes se lisent d'un bloc) — voir la section 4 pour le contexte technique sous-jacent.

---

## 1. Le corpus RecipeML

Le corpus ci-dessous contient **7 recettes** variées : entrées, plats et desserts, avec un mélange volontairement pédagogique de coûts (bon marché / cher), de teneurs en protéines (faible / élevée), d'allergènes (aucun / fruits à coque / gluten / lactose). Chaque recette porte sa fiche nutritionnelle complète et sa liste d'allergènes.

La structure suit la grammaire RecipeML 2.0 (simplifiée) :

```
<recipeml>
  <recipe>
    <head><title>...</title><category>...</category></head>
    <nutrition kcal="..." protein="..." carbs="..." fat="..."/>
    <prep time="..."/>
    <ingredients>...</ingredients>
    <allergens>...</allergens>
    <cost currency="EUR">...</cost>
  </recipe>
</recipeml>
```

In [2]:
// Corpus RecipeML : 7 recettes réparties en entrées / plats / desserts.
// Mix pédagogique : coûts, protéines, allergènes (aucun / noix / gluten / lactose) variés.
string recipesXml = @"<recipeml>
  <recipe>
    <head><title>Salade de quinoa</title><category>entree</category></head>
    <nutrition kcal=""180"" protein=""6"" carbs=""24"" fat=""6""/>
    <prep time=""15""/>
    <ingredients>quinoa, tomate, concombre, citron</ingredients>
    <allergens>none</allergens>
    <cost currency=""EUR"">280</cost>
  </recipe>
  <recipe>
    <head><title>Velouté de potiron</title><category>entree</category></head>
    <nutrition kcal=""120"" protein=""4"" carbs=""18"" fat=""3""/>
    <prep time=""20""/>
    <ingredients>potiron, oignon, crème</ingredients>
    <allergens>lactose</allergens>
    <cost currency=""EUR"">220</cost>
  </recipe>
  <recipe>
    <head><title>Poulet rôti aux herbes</title><category>plat</category></head>
    <nutrition kcal=""520"" protein=""45"" carbs=""8"" fat=""32""/>
    <prep time=""60""/>
    <ingredients>poulet, thym, romarin, ail</ingredients>
    <allergens>none</allergens>
    <cost currency=""EUR"">850</cost>
  </recipe>
  <recipe>
    <head><title>Lasagnes bolognaise</title><category>plat</category></head>
    <nutrition kcal=""610"" protein=""28"" carbs=""55"" fat=""30""/>
    <prep time=""75""/>
    <ingredients>pâtes, boeuf, tomate, béchamel, fromage</ingredients>
    <allergens>gluten,lactose</allergens>
    <cost currency=""EUR"">720</cost>
  </recipe>
  <recipe>
    <head><title>Tofu curry coco</title><category>plat</category></head>
    <nutrition kcal=""430"" protein=""22"" carbs=""30"" fat=""24""/>
    <prep time=""35""/>
    <ingredients>tofu, lait de coco, curry, riz</ingredients>
    <allergens>none</allergens>
    <cost currency=""EUR"">540</cost>
  </recipe>
  <recipe>
    <head><title>Brownie aux noix</title><category>dessert</category></head>
    <nutrition kcal=""380"" protein=""6"" carbs=""42"" fat=""22""/>
    <prep time=""45""/>
    <ingredients>chocolat, beurre, sucre, noix, farine</ingredients>
    <allergens>gluten,lactose,nuts</allergens>
    <cost currency=""EUR"">410</cost>
  </recipe>
  <recipe>
    <head><title>Salade de fruits frais</title><category>dessert</category></head>
    <nutrition kcal=""90"" protein=""1"" carbs=""22"" fat=""0""/>
    <prep time=""10""/>
    <ingredients>fraise, banane, orange, kiwi</ingredients>
    <allergens>none</allergens>
    <cost currency=""EUR"">180</cost>
  </recipe>
</recipeml>";

Console.WriteLine($"Corpus chargé : {recipesXml.Length} caractères XML");
var doc = XDocument.Parse(recipesXml);
int n = doc.Root.Elements("recipe").Count();
Console.WriteLine($"Recettes déclarées : {n}");

Corpus chargé : 2242 caractères XML


Recettes déclarées : 7


---

## 2. Parsing LINQ vers des classes de domaine

Le solveur Z3 ne consomme pas de XML : il travaille sur des **expressions et des contraintes**. On projette donc le corpus RecipeML vers une classe de domaine `Recipe` simple, via `XDocument.Parse` + une requête LINQ. Cette étape est **indépendante du solveur** — elle prépare la donnée.

La classe `Recipe` porte exactement les attributs dont le solveur aura besoin pour poser ses contraintes : calories, protéines, glucides, lipides (pour l'agrégation nutritionnelle), coût (pour la contrainte budgétaire), liste d'allergènes (pour l'exclusion) et catégorie (pour l'équilibre du menu).

In [3]:
// Classe de domaine : projection du XML vers un objet que Z3 va consommer.
public class Recipe
{
    public string Name { get; set; }
    public string Category { get; set; }      // entree | plat | dessert
    public int Kcal { get; set; }
    public int Protein { get; set; }          // grammes
    public int Carbs { get; set; }            // grammes
    public int Fat { get; set; }              // grammes
    public int CostCents { get; set; }        // prix en centimes d'euro
    public int PrepMin { get; set; }          // temps de préparation (minutes)
    public string Allergens { get; set; }     // liste CSV : "none" ou "gluten,lactose,nuts"

    public bool HasAllergen(string a) => Allergens != "none" && Allergens.Split(',').Contains(a);
}

// Parsing LINQ to XML -> List<Recipe>
var corpus = doc.Root.Elements("recipe").Select(r => new Recipe
{
    Name = (string)r.Element("head").Element("title"),
    Category = (string)r.Element("head").Element("category"),
    Kcal = (int)r.Element("nutrition").Attribute("kcal"),
    Protein = (int)r.Element("nutrition").Attribute("protein"),
    Carbs = (int)r.Element("nutrition").Attribute("carbs"),
    Fat = (int)r.Element("nutrition").Attribute("fat"),
    CostCents = (int)r.Element("cost"),
    PrepMin = (int)r.Element("prep").Attribute("time"),
    Allergens = (string)r.Element("allergens") ?? "none"
}).ToList();

Console.WriteLine($"Corpus parsé : {corpus.Count} recettes\n");
Console.WriteLine($"{"Nom",-24} {"Cat",-8} {"kcal",4} {"prot",4} {"glu",4} {"lip",4} {"prix",6} {"allergènes",-18}");
Console.WriteLine(new string('-', 84));
foreach (var rc in corpus)
{
    Console.WriteLine($"{rc.Name,-24} {rc.Category,-8} {rc.Kcal,4} {rc.Protein,4} {rc.Carbs,4} {rc.Fat,4} {rc.CostCents,4} c {rc.Allergens,-18}");
}

Corpus parsé : 7 recettes



Nom                      Cat      kcal prot  glu  lip   prix allergènes        


------------------------------------------------------------------------------------


Salade de quinoa         entree    180    6   24    6  280 c none              


Velouté de potiron       entree    120    4   18    3  220 c lactose           


Poulet rôti aux herbes   plat      520   45    8   32  850 c none              


Lasagnes bolognaise      plat      610   28   55   30  720 c gluten,lactose    


Tofu curry coco          plat      430   22   30   24  540 c none              


Brownie aux noix         dessert   380    6   42   22  410 c gluten,lactose,nuts


Salade de fruits frais   dessert    90    1   22    0  180 c none              


### Interpretation : le corpus est prêt pour le solveur

| Catégorie | Nombre | Recettes |
|-----------|--------|----------|
| entree | 2 | Salade de quinoa, Velouté de potiron |
| plat | 3 | Poulet rôti, Lasagnes bolognaise, Tofu curry coco |
| dessert | 2 | Brownie aux noix, Salade de fruits frais |

**Points clés** :

1. **Aucune référence à Z3 dans cette étape**. Le parsing produit une `List<Recipe>` portable — la même qu'on aurait obtenue depuis une base SQL ou un fichier CSV.
2. **Deux recettes sans allergène** (Salade de quinoa, Poulet rôti, Tofu curry coco, Salade de fruits) face à trois qui en contiennent (lactose, gluten, noix) — l'exclusion d'allergène sera donc une **vraie contrainte**, pas triviale.
3. **Le coût varie de 1,80 EUR à 8,50 EUR** : la contrainte budgétaire va forcer des choix réels.

> **Note** : le coût est stocké en **centimes d'euro** (entier) pour rester dans l'arithmétique entière de Z3. Les valeurs réelles seront divisées par 100 à l'affichage.

---

## 3. Modélisation DSL Z3.Linq — menu équilibré sans allergène

C'est la **section phare** du notebook : on démontre que le DSL Z3.Linq exprime naturellement un menu équilibré **alimenté par les données externes**.

### Le modèle

On associe une **variable booléenne** à chaque recette du corpus (`sel_i = true` si la recette i est retenue). Les contraintes sont :

- **Cardinalité** : exactement une entrée + un plat + un dessert sélectionnés (exclusion mutuelle par catégorie).
- **Budget** : somme des coûts des recettes sélectionnées ≤ plafond (1500 centimes, soit 15 EUR).
- **Protéines** : somme des protéines ≥ 30 g (menu consistant).
- **Exclusion d'allergène** : aucune recette sélectionnée ne contient de fruits à coque (`nuts`).
- **Calories** : total entre 500 et 900 kcal (équilibre énergétique).

### Clé technique : agrégation conditionnelle par `MkITE`

Comme dans le notebook 05 section 5, la nutrition totale est une somme de termes conditionnels `MkITE(sel_i, valeur_i, 0)` — chaque recette contribue à la somme si et seulement si elle est sélectionnée. Les **valeurs** (`Kcal`, `Protein`, `CostCents`) proviennent du corpus RecipeML, mais l'**expression Z3** est identique au cas littéral : c'est tout l'intérêt de la fusion de données.

In [4]:
// Modèle booléen Z3 : un BoolExpr par recette du corpus, agrégation nutritionnelle via MkITE.
// Valeurs (Kcal, Protein, CostCents) proviennent du XML parsé -> data fusion externe.

using (var z3ctx = new Context())
{
    var solver = z3ctx.MkSolver();

    // Une variable booléenne par recette, indexée par sa position dans le corpus.
    var selections = corpus
        .Select((rc, i) => new { Recipe = rc, Index = i, Var = (BoolExpr)z3ctx.MkConst($"sel_{i}_{rc.Name}", z3ctx.BoolSort) })
        .ToList();

    // Cardinalité par catégorie : exactement 1 sélectionné dans chaque catégorie.
    void ExactlyOne(string cat)
    {
        var catVars = selections.Where(s => s.Recipe.Category == cat).Select(s => (BoolExpr)s.Var).ToArray();
        solver.Add(z3ctx.MkOr(catVars));                              // au moins 1
        for (int a = 0; a < catVars.Length; a++)                       // au plus 1 (exclusion par paires)
            for (int b = 0; b < catVars.Length; b++)
                if (a != b)
                    solver.Add(z3ctx.MkImplies(catVars[a], z3ctx.MkNot(catVars[b])));
    }
    ExactlyOne("entree");
    ExactlyOne("plat");
    ExactlyOne("dessert");

    // Agrégations nutritionnelles : somme de MkITE (contribution conditionnelle).
    IntExpr totalKcal = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0),
        (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Recipe.Kcal), z3ctx.MkInt(0))));
    IntExpr totalProt = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0),
        (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Recipe.Protein), z3ctx.MkInt(0))));
    IntExpr totalCost = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0),
        (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Recipe.CostCents), z3ctx.MkInt(0))));

    // Contraintes scalaires : équilibre énergétique, apport protéique, budget.
    solver.Add(z3ctx.MkGe(totalKcal, z3ctx.MkInt(500)));
    solver.Add(z3ctx.MkLe(totalKcal, z3ctx.MkInt(900)));
    solver.Add(z3ctx.MkGe(totalProt, z3ctx.MkInt(30)));
    solver.Add(z3ctx.MkLe(totalCost, z3ctx.MkInt(1500)));

    // Exclusion d'allergène : pour chaque recette contenant 'nuts', on interdit sa sélection.
    string bannedAllergen = "nuts";
    foreach (var s in selections.Where(s => s.Recipe.HasAllergen(bannedAllergen)))
        solver.Add(z3ctx.MkNot(s.Var));

    Console.WriteLine($"Contraintes : {solver.NumAssertions} assertions | allergène banni : {bannedAllergen}\n");

    var status = solver.Check();
    Console.WriteLine($"Résultat solve : {status}\n");

    if (status == Status.SATISFIABLE)
    {
        var model = solver.Model;
        int k = 0, p = 0, c = 0;
        Console.WriteLine("Menu équilibré sans fruits à coque :");
        foreach (var s in selections)
        {
            if (model.Eval(s.Var, true).BoolValue == Z3_lbool.Z3_L_TRUE)
            {
                Console.WriteLine($"  [{s.Recipe.Category,-7}] {s.Recipe.Name,-24} {s.Recipe.Kcal,3} kcal | {s.Recipe.Protein,2}g prot | {s.Recipe.CostCents/100.0:F2} EUR | allergènes : {s.Recipe.Allergens}");
                k += s.Recipe.Kcal; p += s.Recipe.Protein; c += s.Recipe.CostCents;
            }
        }
        Console.WriteLine($"  --- TOTAL : {k} kcal | {p}g protéines | {c/100.0:F2} EUR ---");
    }
    else
    {
        Console.WriteLine("Aucun menu ne satisfait les contraintes.");
    }
}

Contraintes : 18 assertions | allergène banni : nuts



Résultat solve : SATISFIABLE



Menu équilibré sans fruits à coque :


  [entree ] Velouté de potiron       120 kcal |  4g prot | 2,20 EUR | allergènes : lactose


  [plat   ] Lasagnes bolognaise      610 kcal | 28g prot | 7,20 EUR | allergènes : gluten,lactose


  [dessert] Salade de fruits frais    90 kcal |  1g prot | 1,80 EUR | allergènes : none


  --- TOTAL : 820 kcal | 33g protéines | 11,20 EUR ---


### Interpretation : ce que le solveur a choisi

Le solver satisfait l'ensemble des contraintes en ignorant explicitement le brownie (il contient des noix). Le `MkITE` **masque entièrement** la valeur des recettes non sélectionnées : leur contribution est littéralement `0` dans chaque agrégat, sans qu'on ait à énumérer les combinaisons.

| Aspect | Rôle du DSL Z3.Linq |
|--------|---------------------|
| Variable par recette | `MkConst` booléen indexé par position corpus |
| Agrégat nutritionnel | `Aggregate` + `MkITE` — **identique** au cas littéral (05 section 5) |
| Exclusion d'allergène | `solver.Add(MkNot(s.Var))` pour chaque recette contenant l'allergène |
| Cardinalité 1/catégorie | `MkOr` + exclusion par paires |

**Pourquoi le DSL est élégant ici** : si l'on voulait la même chose **sans solveur** (notebook 05 section 4, approche par énumération), il faudrait balayer `2 × 3 × 2 = 12` combinaisons à la main, appliquer chaque filtre (calories, protéines, budget, allergène) sur chaque candidat, puis retenir le premier satisfaisant. Avec Z3, **chaque contrainte est déclarée indépendamment**, le solveur se charge de la combinatoire. C'est exactement la distinction entre **programmation déclarative** et **énumération impérative** — et elle devient cruciale dès que le corpus grandit (50 recettes → énumération inenvisageable, solveur immédiat).

> **Note** : la valeur retournée dépend de l'ordre interne du solver. Si vous relancez la cellule avec un budget plus serré, le solveur peut basculer vers le Tofu curry coco (végétarien, 5,40 EUR) à la place du Poulet rôti.

---

## 4. Théorème hiérarchique `int[][]` sur le corpus

On monte d'un cran : au lieu d'un menu unique, on planifie **3 jours × 2 créneaux** (déjeuner, dîner) en sélectionnant des **indices de recettes dans le corpus parsé**. C'est exactement le pattern du notebook 05 section 7, mais avec deux différences :

1. Les bornes d'index ne proviennent plus d'une `foodDatabase` codée à la main mais du **corpus RecipeML** (`corpus`).
2. On impose une **non-répétition** entre les déjeuners via `Z3Methods.Distinct` — le mode `CollectionHandling.Array` explicite, marqueur des colonnes SMT globales (notebook 05 section 7 variante B).

### Contexte technique : le modèle de soumission de .NET Interactive

> La cellule ci-dessous regroupe **dans le même bloc** la déclaration des bornes (`platLo`, `platHi`, etc.), la construction du théorème `Where(...)`, **et** l'appel `.Solve()`. C'est un choix de **lisibilité** : bornes et contraintes se lisent d'un seul tenant.
>
> **Arrière-plan technique** (utile à connaître) : en kernel `.net-csharp`, chaque cellule est compilée dans un assembly dynamique distinct appelé une *submission* (`Submission#0`, `Submission#1`, …). Lorsqu'un lambda de contrainte capture une variable déclarée dans une autre cellule, le runtime doit résoudre cette capture *cross-submission*. Historiquement, le `ExpressionVisitor` du fork Z3.Linq résolvait ces constantes par réflexion (`field.GetValue(target)`), ce qui échouait avec une `ArgumentException : field defined on Submission#N is not a field on Submission#M`.
>
> **Ce bug est désormais résolu** : le correctif apporté à `Theorem.AssertConstraints` (réplique de l'issue fork #2) replie les constantes capturées en littéraux via `PartialEvaluator.PartialEval` **avant** la visite de l'arbre d'expression, neutralisant la résolution réflexive cross-assembly. La capture d'une variable déclarée dans une cellule précédente fonctionne donc **désormais** — un test de régression polyglot le valide (`CrossSubmissionCaptureRepro.ipynb`).
>
> Garder bornes + contraintes + solve dans la même cellule reste une bonne pratique de **lisibilité** (et facilite le débogage d'un modèle isolé), mais ce n'est plus une exigence technique.

In [5]:
// Théorème hiérarchique int[][] sur le corpus RecipeML.
// Recommandation de lisibilité : bornes + Where(...) + Solve() dans la MÊME cellule.
// (La capture cross-submission est désormais supportée par le correctif AssertConstraints.)

public class DayPlan
{
    public int[][] Sel { get; set; } = new int[3][];   // 3 jours x 2 créneaux (0=déjeuner, 1=dîner)
    public DayPlan() { for (int t = 0; t < 3; t++) Sel[t] = new int[2]; }
}

// Bornes d'index par catégorie dans le corpus parsé (et PAS en dur).
int entLo = corpus.FindIndex(r => r.Category == "entree");
int entHi = corpus.FindLastIndex(r => r.Category == "entree");
int pltLo = corpus.FindIndex(r => r.Category == "plat");
int pltHi = corpus.FindLastIndex(r => r.Category == "plat");
int desLo = corpus.FindIndex(r => r.Category == "dessert");
int desHi = corpus.FindLastIndex(r => r.Category == "dessert");
Console.WriteLine($"Bornes corpus -> entrées [{entLo},{entHi}], plats [{pltLo},{pltHi}], desserts [{desLo},{desHi}]");

// Pour le plan multi-jours : déjeuner = entrée + plat, dîner = plat + dessert.
// On impose que chaque déjeuner soit dans [platLo, platHi] (simplification pédagogique).
using (var ctx = new Z3Context { DefaultCollectionHandling = CollectionHandling.Array })
{
    var th = ctx.NewTheorem<DayPlan>();

    // Contrainte de catégorie sur chaque cellule du plan (bounds corpus -> pur LINQ).
    for (int t = 0; t < 3; t++)
    {
        int tt = t;
        th = th.Where(g => g.Sel[tt][0] >= pltLo && g.Sel[tt][0] <= pltHi);   // déjeuner = plat
        th = th.Where(g => g.Sel[tt][1] >= pltLo && g.Sel[tt][1] <= pltHi);   // dîner    = plat
    }

    // Non-répétition des déjeuners sur les 3 jours (Distinct cross-row, mode Array explicite).
    th = th.Where(g => Z3Methods.Distinct(g.Sel[0][0], g.Sel[1][0], g.Sel[2][0]));

    var sw = System.Diagnostics.Stopwatch.StartNew();
    DayPlan sol = th.Solve();
    sw.Stop();

    Console.WriteLine($"\nThéorème hiérarchique int[][] résolu en {sw.Elapsed.TotalMilliseconds:F1} ms (status : {(sol != null ? "SAT" : "UNSAT")})\n");

    if (sol != null)
    {
        for (int t = 0; t < 3; t++)
        {
            var lunch = corpus[sol.Sel[t][0]];
            var dinner = corpus[sol.Sel[t][1]];
            Console.WriteLine($"Jour {t+1} : déjeuner={lunch.Name,-24} | dîner={dinner.Name,-24}");
        }
    }
    else
    {
        Console.WriteLine("(Aucun plan réalisable avec ces bornes.)");
    }
}

Bornes corpus -> entrées [0,1], plats [2,4], desserts [5,6]



Théorème hiérarchique int[][] résolu en 65,6 ms (status : SAT)



Jour 1 : déjeuner=Lasagnes bolognaise      | dîner=Poulet rôti aux herbes  


Jour 2 : déjeuner=Poulet rôti aux herbes   | dîner=Tofu curry coco         


Jour 3 : déjeuner=Tofu curry coco          | dîner=Lasagnes bolognaise     


### Interpretation : non-répétition comme contrainte globale SMT

Le solveur trouve un plan où les **trois déjeuners sont des plats différents** du corpus. Le `Z3Methods.Distinct` posé sur les accès `g.Sel[0][0]`, `g.Sel[1][0]`, `g.Sel[2][0]` est traduit par le ré-écriveur LINQ en une contrainte `distinct` SMT-LIB native — **pas** une expansion en comparaisons par paires `a ≠ b ∧ a ≠ c ∧ b ≠ c` que le programmeur devrait écrire lui-même.

| Élément | Implémentation |
|---------|----------------|
| Stockage 3×2 | `int[][] Sel` initialisé dans le constructeur (sinon NRE après `Solve`) |
| Bornes de catégorie | Calculées depuis `corpus` (pas codées en dur) |
| Non-répétition | `Z3Methods.Distinct` sur accès cross-row du tableau imbriqué |
| Mode | `CollectionHandling.Array` explicite (marqueur SMT-LIB `distinct`) |

**Points clés** :

1. Le **même pattern `int[][]`** qui structure un Sudoku (notebook 04) ou un plan hebdomadaire (notebook 05 section 7) fonctionne à l'identique sur un corpus externe — la source de données est orthogonale au modèle.
2. L'organisation « tout dans la même cellule » est un choix de **lisibilité** : elle n'impose aucune limite d'expressivité au modèle. Depuis le correctif `AssertConstraints` (capture cross-submission désormais supportée), les contraintes peuvent aussi bien référencer des variables déclarées dans une cellule précédente — le code reste déclaratif.
3. Avec 3 plats dans le corpus et une contrainte de 3 déjeuners distincts, le solver fait une **permutation complète** de la catégorie `plat` — un cas de non-trivialité Prong-B : la contrainte `Distinct` accomplit ici un travail qu'une simple borne ne pourrait pas exprimer.

---

## 5. Tableau récapitulatif

| Section | Modèle | Technique clé | Source des données |
|---------|--------|---------------|--------------------|
| 2 | Parsing LINQ to XML | `XDocument.Parse` → `List<Recipe>` | Chaîne XML RecipeML littérale |
| 3 | Sélection booléenne avec allergène exclu | `MkConst` booléen + `MkITE` + `MkNot` | Valeurs nutritionnelles du corpus parsé |
| 4 | Plan hiérarchique multi-jours | `int[][]` + `Z3Methods.Distinct` + `CollectionHandling.Array` | Bornes d'index issues du corpus parsé |

### Comparaison avec le notebook 05

| Aspect | Notebook 05 (littéral) | Notebook 06b (RecipeML) |
|--------|------------------------|--------------------------|
| Source des données | `List<FoodItem>` codée main | XML externe parsé par `XDocument` |
| Modèle section phare | Booléen + `MkITE` | Identique + exclusion d'allergène |
| Modèle hiérarchique | `WeeklyPlan` 3×3 | `DayPlan` 3×2 avec `Distinct` |
| Élément distinctif | Data fusion | **Fusion depuis un standard XML** |

---

## Exercices

Les exercices ci-dessous étendent le modèle sur le corpus RecipeML. Chaque stub est volontairement incomplet : remplacez les commentaires `// TODO etudiant` par votre code. Les stubs ne lèvent **pas** d'exception — le notebook s'exécute de bout en bout même exercices non remplis.

### Exercice 1 : Contrainte de glucides dans le menu booléen

Reprenez le modèle de la section 3 et ajoutez une contrainte sur les **glucides totaux** : le menu final doit contenir **entre 40 g et 90 g** de glucides. Le brownie reste exclu.

**Indices** :
- `# Etape 1` : déclarez un `IntExpr totalCarbs` avec le même pattern `Aggregate` + `MkITE` que `totalKcal`, mais en utilisant `s.Recipe.Carbs`.
- `# Etape 2` : ajoutez `solver.Add(z3ctx.MkGe(totalCarbs, z3ctx.MkInt(40)))` et la borne supérieure (90).
- `# Etape 3` : affichez la valeur des glucides dans le menu final pour vérifier.

In [6]:
// Exercice 1 : contrainte de glucides (40g <= total <= 90g) sur le modèle de la section 3.
// TODO etudiant : reprendre la cellule de la section 3 et ajouter totalCarbs + ses bornes.

// Indice :
// IntExpr totalCarbs = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0),
//     (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Recipe.Carbs), z3ctx.MkInt(0))));
// solver.Add(z3ctx.MkGe(totalCarbs, z3ctx.MkInt(40)));
// solver.Add(z3ctx.MkLe(totalCarbs, z3ctx.MkInt(90)));

Console.WriteLine("Exercice à compléter : contrainte de glucides sur le menu booléen (section 3)");

Exercice à compléter : contrainte de glucides sur le menu booléen (section 3)


### Exercice 2 : Menu à protéines maximales sous budget (dichotomie)

Le solveur `MkSolver` trouve *une* solution satisfaisante mais pas nécessairement la **meilleure**. Étendez le modèle de la section 3 pour trouver le menu qui **maximise les protéines** tout en respectant le budget de 15 EUR. Utilisez le pattern de **dichotomie sur le plancher de protéines** (notebook 05 section 6).

**Indices** :
- `# Etape 1` : posez `floor = 0`, `ceil = 200` (borne haute large sur les protéines).
- `# Etape 2` : boucle `while (ceil - floor > 1)` : testez `mid = (floor + ceil) / 2`, ajoutez `solver.Add(MkGe(totalProt, MkInt(mid)))`, si SAT conservez `floor = mid` (et le modèle), sinon `ceil = mid`. Utilisez `solver.Push()` / `solver.Pop()` entre itérations.
- `# Etape 3` : affichez le menu optimal et la valeur de protéines atteinte.

In [7]:
// Exercice 2 : maximisation des protéines sous budget (dichotomie, cf. notebook 05 section 6).
// TODO etudiant : reprendre le modèle booléen et remplacer la résolution unique par une dichotomie.

// Indice :
// int floor = 0, ceil = 200;
// while (ceil - floor > 1) {
//     solver.Push();
//     int mid = (floor + ceil) / 2;
//     solver.Add(z3ctx.MkGe(totalProt, z3ctx.MkInt(mid)));
//     if (solver.Check() == Status.SATISFIABLE) floor = mid; else { ceil = mid; solver.Pop(); }
// }

Console.WriteLine("Exercice à compléter : menu à protéines maximales sous budget");

Exercice à compléter : menu à protéines maximales sous budget


### Exercice 3 : Menu végétarien uniquement

Le corpus contient au moins une recette végétarienne par catégorie (Salade de quinoa, Velouté de potiron, Tofu curry coco, Salade de fruits frais). Ajoutez un **filtre végétarien** : aucune recette contenant de la viande ne peut être sélectionnée.

**Indices** :
- `# Etape 1` : marquez les recettes non végétariennes. Pour cet exercice, on considère comme **non-végétariennes** les recettes dont le nom contient `"Poulet"`, `"boeuf"` ou `"Lasagnes"` (heuristic simple — un vrai système utiliserait un champ `<diet>` du XML).
- `# Etape 2` : pour chaque recette non végétarienne, ajoutez `solver.Add(z3ctx.MkNot(s.Var))` (même pattern que l'exclusion d'allergène).
- `# Etape 3` : vérifiez que le solveur choisit bien le Tofu curry coco comme plat.

In [8]:
// Exercice 3 : menu végétarien uniquement (filtre par nom de recette).
// TODO etudiant : exclure les recettes non-végétariennes du modèle booléen.

// Indice :
// string[] nonVege = new[] { "Poulet", "boeuf", "Lasagnes" };
// foreach (var s in selections)
// {
//     bool isNonVege = nonVege.Any(k => s.Recipe.Name.Contains(k));
//     if (isNonVege) solver.Add(z3ctx.MkNot(s.Var));
// }

Console.WriteLine("Exercice à compléter : menu végétarien uniquement");

Exercice à compléter : menu végétarien uniquement


---

## Conclusion

Ce notebook a démontré que le DSL Z3.Linq fonctionne **à l'identique** que les données proviennent d'un littéral en mémoire (notebook 05) ou d'un **document structuré externe au format standard RecipeML**. La frontière entre la donnée et le solveur est franchie au moment du **parsing LINQ to XML** ; ensuite, les modèles Z3 sont réutilisés tels quels.

### Points clés à retenir

| Concept | Implémentation |
|---------|----------------|
| Parsing standard XML | `XDocument.Parse` + projection `Select` vers `List<Recipe>` |
| Sélection booléenne + allergène | `MkNot(s.Var)` pour chaque recette contenant l'allergène banni |
| Agrégation nutritionnelle | `Aggregate` + `MkITE` — valeurs du corpus, expression Z3 standard |
| Plan multi-jours non-répétitif | `int[][]` + `Z3Methods.Distinct` + `CollectionHandling.Array` explicite |
| Organisation du code | Bornes + `Where(...)` + `Solve()` regroupés pour la **lisibilité** (capture cross-submission désormais supportée par le correctif `AssertConstraints`) |

### Perspective

La même architecture (parser → classe de domaine → théorème Z3.Linq) se transpose à n'importe quelle source de données : base SQL via Entity Framework, API REST via `HttpClient`, fichier CSV via `CsvHelper`. Le solveur reste découplé de l'origine des données — c'est ce qui fait la force du **paradigme déclaratif** en optimisation sous contraintes.

**Navigation** : [Index Z3](./README.md) | [Notebook 05 (Meal Planner)](./05_Meal_Planner_Hierarchical.ipynb)